#主成分分析により多重共線性の解消を目指す

In [1]:
#import
import pandas as pd
from sklearn.model_selection import KFold #K分割交差検証
from sklearn.model_selection import cross_validate  #K分割交差検証

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰

from statsmodels.stats.outliers_influence import variance_inflation_factor  #VIF
from sklearn.decomposition import PCA  #PCA

In [2]:
#データフレームの読み込み
'''
df1 VIFの高さを解消するために一部の列を消去する前のデータフレーム
df2 〃した後のデータフレーム

以降は基本的にdf1を用いて分析を行う。
sc_x,df_yはdf1を基に作成する。
ただし、必要があればdf1も利用する。
'''
df1 = pd.read_csv('datafiles/df1_all_col.csv')

sc_x = df1.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df1['SalePrice'])

In [3]:
#K分割交差検証のための準備
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [4]:
#モデルの作成
#リッジ回帰
model2 = Ridge(alpha = 100)
model2.fit(sc_x, df_y)

,alpha,100
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [5]:
#THE_COL_NAME　は 'GrandFinish' のようにダミー変数化したい列の名
#ダミー変数化されたTHE_COL_NAME列に主成分分析を実施し、変数を減らしたsc_x1を返す関数（非破壊関数）
def change_to_PCA(THE_COL_NAME, sc_x):
    sc_x1 = sc_x.copy()

    if any((THE_COL_NAME in c) for c in sc_x1.columns):
        #THE_COL_NAME列をダミー変数化した列一覧をリスト化し、主成分分析により一列化
        THE_COL_NAME_cols = []
        for c in sc_x1.columns:
            if THE_COL_NAME in c:
                THE_COL_NAME_cols.append(c)
        print(f'削除した列一覧＝{THE_COL_NAME_cols}')
        print(f'削除した列数＝{len(THE_COL_NAME_cols)}')



        #累積寄与率の閾値を0.8として、n_componentsを設定
        #適切なPCAのための特徴量数の設定
        PCAmodel = PCA(whiten = True)
        THE_COL_NAME_df = pd.DataFrame()
        for c in THE_COL_NAME_cols:
            THE_COL_NAME_df = pd.concat([THE_COL_NAME_df, sc_x1[c]], axis = 1)
        PCAmodel.fit(THE_COL_NAME_df)

        thred = 0.8
        final_num = 0
        ratio =PCAmodel.explained_variance_ratio_
        for i in range(len(ratio)):
            ruiseki = sum(ratio[0:i+1])    #i+1個めの特徴量までの累積寄与率
            if ruiseki >= thred:   #i+1個めの特徴量において初めて累積寄与率がthredを超えるならば
                final_num = i + 1   #特徴量はi+1個必要である
                break
        print(f'PCAにより作成された特徴量数＝{final_num}')



        #最適な特徴量数で、主成分分析の実施
        PCAmodel = PCA(n_components = final_num, whiten = True)
        PCAmodel.fit(THE_COL_NAME_df)

        #主成分によるデータフレームをTHE_COL_NAME_PCA_dfとする
        THE_COL_NAME_PCA = PCAmodel.transform(THE_COL_NAME_df)
        THE_COL_NAME_PCA_df = pd.DataFrame(THE_COL_NAME_PCA)

        col_name = []
        for i in range(final_num):
            name = THE_COL_NAME + '_PCA_' + str(i)
            col_name.append(name)
        THE_COL_NAME_PCA_df.columns = col_name



            #主成分分析により作成した列を、実施前の列と置き換える
        for c in sc_x1[THE_COL_NAME_cols]:
            sc_x1 = sc_x1.drop([c], axis = 1)
        sc_x1 = pd.concat([sc_x1, THE_COL_NAME_PCA_df], axis = 1)



        return sc_x1


    
    else:
        print(f'{THE_COL_NAME}を含む列はありません')
        return 0



In [6]:
sc_x1 = change_to_PCA('GarageFinish', sc_x)

#スコアの確認
model2.fit(sc_x1, df_y)
result = cross_validate(model2, sc_x1, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

削除した列一覧＝['GarageFinish_NA', 'GarageFinish_RFn', 'GarageFinish_Unf']
削除した列数＝3
PCAにより作成された特徴量数＝2
完成したmodel2のスコア＝0.7776795508943494


In [7]:
'''
実施前
完成したmodel2のスコア＝0.7774453479883086

実施後
完成したmodel2のスコア＝0.7776795508943494

よって誤差程度の改善が見られた。
'''

'\n実施前\n完成したmodel2のスコア＝0.7774453479883086\n\n実施後\n完成したmodel2のスコア＝0.7776795508943494\n\nよって誤差程度の改善が見られた。\n'

In [8]:
#VIFがinfであった列をすべて主成分分析
to_PCA = ['Bsmt', 'FlrSF', 'Exterior2nd', 'Exterior1st']
sc_x2 = sc_x.copy()
for c in to_PCA:
    sc_x2 = change_to_PCA(c, sc_x2)
    
#スコアの確認
model2.fit(sc_x2, df_y)
result = cross_validate(model2, sc_x2, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

削除した列一覧＝['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtFinType1_BLQ', 'BsmtFinType1_GLQ', 'BsmtFinType1_LwQ', 'BsmtFinType1_Rec', 'BsmtFinType1_Unf', 'BsmtQual_Fa', 'BsmtQual_Gd', 'BsmtQual_NA', 'BsmtQual_TA', 'BsmtFinType2_BLQ', 'BsmtFinType2_GLQ', 'BsmtFinType2_LwQ', 'BsmtFinType2_Rec', 'BsmtFinType2_Unf', 'BsmtCond_Gd', 'BsmtCond_Po', 'BsmtCond_TA', 'BsmtExposure_Gd', 'BsmtExposure_Mn', 'BsmtExposure_No']
削除した列数＝26
PCAにより作成された特徴量数＝13
削除した列一覧＝['1stFlrSF', '2ndFlrSF']
削除した列数＝2
PCAにより作成された特徴量数＝2
削除した列一覧＝['Exterior2nd_AsphShn', 'Exterior2nd_Brk Cmn', 'Exterior2nd_BrkFace', 'Exterior2nd_CBlock', 'Exterior2nd_CmentBd', 'Exterior2nd_HdBoard', 'Exterior2nd_ImStucc', 'Exterior2nd_MetalSd', 'Exterior2nd_Other', 'Exterior2nd_Plywood', 'Exterior2nd_Stone', 'Exterior2nd_Stucco', 'Exterior2nd_VinylSd', 'Exterior2nd_Wd Sdng', 'Exterior2nd_Wd Shng']
削除した列数＝15
PCAにより作成された特徴量数＝12
削除した列一覧＝['Exterior1st_AsphShn', 'Exterior1st_BrkComm', 'Exterior1st_BrkFace

In [9]:
'''
実施前
完成したmodel2のスコア＝0.7774453479883086

実施後
完成したmodel2のスコア＝0.7861392211115689

よって誤差程度の改善が見られた。
'''

'\n実施前\n完成したmodel2のスコア＝0.7774453479883086\n\n実施後\n完成したmodel2のスコア＝0.7861392211115689\n\nよって誤差程度の改善が見られた。\n'

In [10]:
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x2, i) for i in range(sc_x2.shape[1])]
vif_df["features"]=sc_x2.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

c:\Users\natsu\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,VIF_Factor,features
8,inf,LowQualFinSF
9,inf,GrLivArea
208,inf,FlrSF_PCA_0
209,inf,FlrSF_PCA_1
192,2163.866979,GarageFinish_NA
16,2142.848802,GarageYrBlt
41,942.104303,MiscFeature_NA
43,787.715181,MiscFeature_Shed
120,280.231686,GarageCond_TA
106,232.113571,GarageQual_TA


In [11]:
#先ほどのsc_x2をさらにPCAをかけて改善する
to_PCA = ['FinSF', 'GarageFinish', 'MiscFeature', 'GarageCond', 'GarageQual', 'RoofStyle']
for c in to_PCA:
    sc_x2 = change_to_PCA(c, sc_x2)
    print()
    
#スコアの確認
model2.fit(sc_x2, df_y)
result = cross_validate(model2, sc_x2, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

削除した列一覧＝['LowQualFinSF']
削除した列数＝1
PCAにより作成された特徴量数＝1

削除した列一覧＝['GarageFinish_NA', 'GarageFinish_RFn', 'GarageFinish_Unf']
削除した列数＝3
PCAにより作成された特徴量数＝2

削除した列一覧＝['MiscFeature_NA', 'MiscFeature_Othr', 'MiscFeature_Shed', 'MiscFeature_TenC']
削除した列数＝4
PCAにより作成された特徴量数＝3

削除した列一覧＝['GarageCond_Fa', 'GarageCond_Gd', 'GarageCond_Po', 'GarageCond_TA']
削除した列数＝4
PCAにより作成された特徴量数＝3

削除した列一覧＝['GarageQual_Fa', 'GarageQual_Gd', 'GarageQual_Po', 'GarageQual_TA']
削除した列数＝4
PCAにより作成された特徴量数＝3

削除した列一覧＝['RoofStyle_Gable', 'RoofStyle_Gambrel', 'RoofStyle_Hip', 'RoofStyle_Mansard', 'RoofStyle_Shed']
削除した列数＝5
PCAにより作成された特徴量数＝4

完成したmodel2のスコア＝0.7863174984490457


In [12]:
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x2, i) for i in range(sc_x2.shape[1])]
vif_df["features"]=sc_x2.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 50]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

c:\Users\natsu\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,VIF_Factor,features
8,inf,GrLivArea
187,inf,FlrSF_PCA_0
188,inf,FlrSF_PCA_1
212,inf,FinSF_PCA_0
82,89.880508,ExterCond_TA
101,80.078922,GarageType_Attchd
80,75.887027,ExterCond_Gd
115,65.892349,RoofMatl_CompShg
105,64.504048,GarageType_Detchd
15,58.504251,GarageYrBlt


In [13]:
#先ほどのsc_x2をさらにPCAをかけて改善する
to_PCA = ['ExterCond', 'GarageType', 'RoofMatl', 'SaleType']
for c in to_PCA:
    sc_x2 = change_to_PCA(c, sc_x2)
    print()
    
#スコアの確認
model2.fit(sc_x2, df_y)
result = cross_validate(model2, sc_x2, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')


#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x2, i) for i in range(sc_x2.shape[1])]
vif_df["features"]=sc_x2.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

削除した列一覧＝['ExterCond_Fa', 'ExterCond_Gd', 'ExterCond_Po', 'ExterCond_TA']
削除した列数＝4
PCAにより作成された特徴量数＝3

削除した列一覧＝['GarageType_Attchd', 'GarageType_Basment', 'GarageType_BuiltIn', 'GarageType_CarPort', 'GarageType_Detchd']
削除した列数＝5
PCAにより作成された特徴量数＝4

削除した列一覧＝['RoofMatl_CompShg', 'RoofMatl_Membran', 'RoofMatl_Metal', 'RoofMatl_Roll', 'RoofMatl_Tar&Grv', 'RoofMatl_WdShake', 'RoofMatl_WdShngl']
削除した列数＝7
PCAにより作成された特徴量数＝5

削除した列一覧＝['SaleType_CWD', 'SaleType_Con', 'SaleType_ConLD', 'SaleType_ConLI', 'SaleType_ConLw', 'SaleType_New', 'SaleType_Oth', 'SaleType_WD']
削除した列数＝8
PCAにより作成された特徴量数＝6

完成したmodel2のスコア＝0.7831956444285624


c:\Users\natsu\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,VIF_Factor,features
8,inf,GrLivArea
188,inf,FinSF_PCA_0
163,inf,FlrSF_PCA_0
164,inf,FlrSF_PCA_1
36,48.374725,MSZoning_RL
15,40.818784,GarageYrBlt
98,38.534718,Heating_GasA
190,35.449367,GarageFinish_PCA_1
0,34.232989,MSSubClass
177,33.561042,Exterior1st_PCA_0


In [ ]:
#実験的にVIFが大きい列を削除してみる
sc_x3 = sc_x2.copy()
to_drop = ['GrLivArea', 'FinSF_PCA_0', 'FlrSF_PCA_1']
for c in to_drop:
    sc_x3 = sc_x3.drop([c], axis = 1)
 
#スコアの確認
model2.fit(sc_x3, df_y)
result = cross_validate(model2, sc_x3, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')


#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x3, i) for i in range(sc_x3.shape[1])]
vif_df["features"]=sc_x3.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

完成したmodel2のスコア＝0.7951580267152719


,VIF_Factor,features
35,48.319608,MSZoning_RL
14,40.688963,GarageYrBlt
97,38.451543,Heating_GasA
187,35.407118,GarageFinish_PCA_1
0,34.208681,MSSubClass
175,33.552640,Exterior1st_PCA_0
163,33.395432,Exterior2nd_PCA_0
36,32.460400,MSZoning_RM
91,32.016647,MasVnrType_NA
90,27.664057,MasVnrType_BrkFace


In [17]:
#スコア0.01程度の精度の向上が見られた。さらに、VIFが大幅に低下した。
#引き続き、sc_x3に対してさらにPCAをかけて改善する
to_PCA = ['MSZoning', 'Heating', 'MasVnrType', 'Exterior']
for c in to_PCA:
    sc_x3 = change_to_PCA(c, sc_x3)
    print()

#スコアの確認
model2.fit(sc_x3, df_y)
result = cross_validate(model2, sc_x3, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')


#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x3, i) for i in range(sc_x3.shape[1])]
vif_df["features"]=sc_x3.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

削除した列一覧＝['MSZoning_FV', 'MSZoning_RH', 'MSZoning_RL', 'MSZoning_RM']
削除した列数＝4
PCAにより作成された特徴量数＝3

削除した列一覧＝['HeatingQC_Fa', 'HeatingQC_Gd', 'HeatingQC_Po', 'HeatingQC_TA', 'Heating_GasA', 'Heating_GasW', 'Heating_Grav', 'Heating_OthW', 'Heating_Wall']
削除した列数＝9
PCAにより作成された特徴量数＝6

削除した列一覧＝['MasVnrType_BrkFace', 'MasVnrType_NA', 'MasVnrType_Stone']
削除した列数＝3
PCAにより作成された特徴量数＝2

削除した列一覧＝['Exterior2nd_PCA_0', 'Exterior2nd_PCA_1', 'Exterior2nd_PCA_2', 'Exterior2nd_PCA_3', 'Exterior2nd_PCA_4', 'Exterior2nd_PCA_5', 'Exterior2nd_PCA_6', 'Exterior2nd_PCA_7', 'Exterior2nd_PCA_8', 'Exterior2nd_PCA_9', 'Exterior2nd_PCA_10', 'Exterior2nd_PCA_11', 'Exterior1st_PCA_0', 'Exterior1st_PCA_1', 'Exterior1st_PCA_2', 'Exterior1st_PCA_3', 'Exterior1st_PCA_4', 'Exterior1st_PCA_5', 'Exterior1st_PCA_6', 'Exterior1st_PCA_7', 'Exterior1st_PCA_8', 'Exterior1st_PCA_9', 'Exterior1st_PCA_10']
削除した列数＝23
PCAにより作成された特徴量数＝11

完成したmodel2のスコア＝0.7946491166329263


,VIF_Factor,features
14,39.924043,GarageYrBlt
148,34.719932,GarageFinish_PCA_1
0,33.575967,MSSubClass
50,26.133940,FireplaceQu_NA
108,22.281787,Neighborhood_NAmes
113,18.192167,Neighborhood_OldTown
91,17.928665,ExterQual_TA
36,16.031446,BldgType_TwnhsE
96,15.933675,SaleCondition_Partial
49,14.862316,FireplaceQu_Gd


In [18]:
#引き続き、sc_x3に対してさらにPCAをかけて改善する
to_PCA = ['FireplaceQu', 'Neighborhood', 'ExterQual', 'BldgType', 'SaleCondition', 'FireplaceQu', 'SaleType']
for c in to_PCA:
    sc_x3 = change_to_PCA(c, sc_x3)
    print()

#スコアの確認
model2.fit(sc_x3, df_y)
result = cross_validate(model2, sc_x3, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')


#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x3, i) for i in range(sc_x3.shape[1])]
vif_df["features"]=sc_x3.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

削除した列一覧＝['FireplaceQu_Fa', 'FireplaceQu_Gd', 'FireplaceQu_NA', 'FireplaceQu_Po', 'FireplaceQu_TA']
削除した列数＝5
PCAにより作成された特徴量数＝4

削除した列一覧＝['Neighborhood_Blueste', 'Neighborhood_BrDale', 'Neighborhood_BrkSide', 'Neighborhood_ClearCr', 'Neighborhood_CollgCr', 'Neighborhood_Crawfor', 'Neighborhood_Edwards', 'Neighborhood_Gilbert', 'Neighborhood_IDOTRR', 'Neighborhood_MeadowV', 'Neighborhood_Mitchel', 'Neighborhood_NAmes', 'Neighborhood_NPkVill', 'Neighborhood_NWAmes', 'Neighborhood_NoRidge', 'Neighborhood_NridgHt', 'Neighborhood_OldTown', 'Neighborhood_SWISU', 'Neighborhood_Sawyer', 'Neighborhood_SawyerW', 'Neighborhood_Somerst', 'Neighborhood_StoneBr', 'Neighborhood_Timber', 'Neighborhood_Veenker']
削除した列数＝24
PCAにより作成された特徴量数＝19

削除した列一覧＝['ExterQual_Fa', 'ExterQual_Gd', 'ExterQual_TA']
削除した列数＝3
PCAにより作成された特徴量数＝2

削除した列一覧＝['BldgType_2fmCon', 'BldgType_Duplex', 'BldgType_Twnhs', 'BldgType_TwnhsE']
削除した列数＝4
PCAにより作成された特徴量数＝4

削除した列一覧＝['SaleCondition_AdjLand', 'SaleCondition_Alloca', 'SaleConditi

,VIF_Factor,features
14,39.167021,GarageYrBlt
107,33.880415,GarageFinish_PCA_1
0,33.304598,MSSubClass
179,24.148821,BldgType_PCA_3
5,13.290814,YearBuilt
86,11.494643,Condition2_Norm
38,10.942612,HouseStyle_1Story


#実験的に、PCAを全行で実施してみる

In [24]:
#PCAをもとのsc_x全列で行うとどうなるのか実験(sc_x4)

sc_x4 = sc_x.copy()
#累積寄与率の閾値を0.8として、n_componentsを設定
#適切なPCAのための特徴量数の設定
PCAmodel = PCA(whiten = True)
PCAmodel.fit(sc_x4)

thred = 0.8
final_num = 0
ratio =PCAmodel.explained_variance_ratio_
for i in range(len(ratio)):
    ruiseki = sum(ratio[0:i+1])    #i+1個めの特徴量までの累積寄与率
    if ruiseki >= thred:   #i+1個めの特徴量において初めて累積寄与率がthredを超えるならば
        final_num = i + 1   #特徴量はi+1個必要である
        break
print(f'PCAにより作成された特徴量数＝{final_num}')



#最適な特徴量数で、主成分分析の実施
PCAmodel = PCA(n_components = final_num, whiten = True)
PCAmodel.fit(sc_x4)


#主成分によるデータフレームをPCA_sc_x4とする
PCA_sc_x4 = PCAmodel.transform(sc_x4)
PCA_sc_x4 = pd.DataFrame(PCA_sc_x4)
col_name = []
for i in range(final_num):
    name = 'PCA_' + str(i)
    col_name.append(name)
PCA_sc_x4.columns = col_name

sc_x4 = pd.concat([sc_x4, PCA_sc_x4], axis = 1)

PCAにより作成された特徴量数＝106


In [25]:
#スコアの確認
model2.fit(sc_x4, df_y)
result = cross_validate(model2, sc_x4, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')


#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x4, i) for i in range(sc_x4.shape[1])]
vif_df["features"]=sc_x4.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

完成したmodel2のスコア＝0.7642168009960459


,VIF_Factor,features
137,7.835046e+06,Condition1_PosN
136,5.345228e+06,Condition1_PosA
74,4.948372e+06,Exterior2nd_VinylSd
336,4.400557e+06,PCA_84
15,3.549176e+06,GrLivArea
...,...,...
329,1.462283e+04,PCA_77
37,1.151243e+04,BsmtFinType1_GLQ
167,8.503400e+03,MasVnrType_BrkFace
34,7.565130e+03,MoSold


In [28]:
df_corr = pd.DataFrame(sc_x4.corr())
display(df_corr['PCA_0'].sort_values(ascending = False))
display(df_corr['PCA_0'].sort_values(ascending = True))

PCA_0                1.000000
YearBuilt            0.832381
OverallQual          0.804344
Foundation_PConc     0.726949
GarageCars           0.723723
                       ...   
GarageType_Detchd   -0.501409
GarageFinish_Unf    -0.569546
KitchenQual_TA      -0.639867
BsmtQual_TA         -0.654598
ExterQual_TA        -0.744522
Name: PCA_0, Length: 358, dtype: float64

ExterQual_TA        -0.744522
BsmtQual_TA         -0.654598
KitchenQual_TA      -0.639867
GarageFinish_Unf    -0.569546
GarageType_Detchd   -0.501409
                       ...   
GarageCars           0.723723
Foundation_PConc     0.726949
OverallQual          0.804344
YearBuilt            0.832381
PCA_0                1.000000
Name: PCA_0, Length: 358, dtype: float64

In [ ]:
#全行にPCAを行っても改善は見られなかった。